# Prompt Engineering Techniques (Các kỹ thuật viết Prompt)

**Khóa: Claude with the Anthropic API (Anthropic Skilljar)**

*Prompt Engineering* là nghệ thuật + khoa học viết chỉ dẫn cho Claude sao cho nó trả lời **đúng – đủ – đúng định dạng** bạn cần. Notebook này đi qua các kỹ thuật cốt lõi, từ cơ bản đến nâng cao.

> 🔑 Nguyên tắc vàng: **Claude không đọc được suy nghĩ của bạn.** Bạn càng rõ ràng, kết quả càng tốt.


In [ ]:
!pip install anthropic

In [ ]:
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

# Hàm tiện ích: gửi prompt và in kết quả
def chat(prompt, system=None, **kwargs):
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=system if system else anthropic.NOT_GIVEN,
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
    )
    return response.content[0].text

## 0. Quy trình Prompt Engineering (Vòng lặp cải tiến)

Prompt engineering không phải "viết 1 lần là xong" mà là một **vòng lặp**:

```
Viết prompt  →  Chạy thử  →  Đánh giá kết quả  →  Sửa prompt  →  (lặp lại)
```

Kết hợp với phần *Prompt Evaluation* (notebook trước) để đo lường khách quan mỗi lần sửa.


## 1. Rõ ràng & Trực tiếp (Be Clear and Direct)

Kỹ thuật quan trọng nhất. Hãy nói **chính xác** điều bạn muốn: định dạng, độ dài, đối tượng đọc, ràng buộc. Đừng để Claude phải đoán.


In [ ]:
# ❌ Mơ hồ
print("--- PROMPT MƠ HỒ ---")
print(chat("Viết về chó."))

print("\n\n--- PROMPT RÕ RÀNG ---")
# ✅ Rõ ràng: nêu rõ định dạng, độ dài, đối tượng
print(chat(
    "Viết một đoạn văn 3 câu giới thiệu giống chó Corgi cho trẻ em 6 tuổi. "
    "Dùng từ ngữ đơn giản, vui tươi."
))

## 2. Gán vai trò (Role Prompting)

Cho Claude một "vai trò" qua **system prompt** → giọng văn, kiến thức và góc nhìn phù hợp hơn. Đây là cách rẻ và hiệu quả để định hình toàn bộ phản hồi.


In [ ]:
system = ("Bạn là một chuyên gia tài chính cá nhân với 20 năm kinh nghiệm. "
          "Trả lời thực tế, dễ hiểu, kèm ví dụ con số cụ thể.")

print(chat("Tôi mới đi làm, nên bắt đầu tiết kiệm thế nào?", system=system))

## 3. Dùng ví dụ (Few-shot / Multishot Prompting)

"Trăm nghe không bằng một thấy." Đưa cho Claude vài **ví dụ mẫu** (input → output) thì nó bắt chước đúng định dạng & phong cách hơn nhiều so với chỉ mô tả bằng lời.

- **Zero-shot**: không ví dụ.
- **Few-shot / Multishot**: vài ví dụ → kết quả ổn định hơn rõ rệt.


In [ ]:
prompt = """
Phân loại cảm xúc của câu thành: TÍCH CỰC, TIÊU CỰC, hoặc TRUNG TÍNH.

Ví dụ:
Câu: "Sản phẩm này tuyệt vời, tôi rất hài lòng!"
Cảm xúc: TÍCH CỰC

Câu: "Giao hàng quá chậm, thất vọng."
Cảm xúc: TIÊU CỰC

Câu: "Hàng đã được giao vào hôm qua."
Cảm xúc: TRUNG TÍNH

Bây giờ phân loại câu sau:
Câu: "Chất lượng ổn nhưng giá hơi cao so với kỳ vọng."
Cảm xúc:""".strip()

print(chat(prompt))

## 4. Dùng thẻ XML để cấu trúc prompt (XML Tags)

Khi prompt có nhiều phần (chỉ dẫn, dữ liệu, ví dụ...), bọc mỗi phần trong **thẻ XML** giúp Claude phân biệt rạch ròi → giảm nhầm lẫn. Claude được huấn luyện đặc biệt tốt với XML tags.


In [ ]:
prompt = """
Hãy tóm tắt văn bản trong thẻ <document> thành đúng 2 câu.

<document>
Trí tuệ nhân tạo (AI) đang thay đổi nhiều ngành nghề. Trong y tế, AI giúp
chẩn đoán bệnh sớm hơn. Trong giáo dục, AI cá nhân hóa việc học cho từng học sinh.
Tuy nhiên, AI cũng đặt ra thách thức về việc làm và đạo đức.
</document>
""".strip()

print(chat(prompt))

## 5. Cho Claude "thời gian suy nghĩ" (Chain of Thought)

Với bài toán phức tạp (toán, logic, suy luận nhiều bước), bắt Claude **suy nghĩ từng bước trước khi trả lời** giúp tăng độ chính xác đáng kể.

**2 cách:**
- **Cách thủ công (khóa cũ dạy):** yêu cầu "suy nghĩ trong thẻ `<thinking>` trước".
- **Cách hiện đại (model mới):** dùng **extended/adaptive thinking** — bật `thinking={"type": "adaptive"}`, Claude tự suy nghĩ sâu.


In [ ]:
# Cách thủ công: yêu cầu suy nghĩ trong thẻ <thinking>
prompt = """
Một cửa hàng giảm giá 20%, rồi giảm thêm 10% trên giá đã giảm.
Một áo giá gốc 500.000đ thì giá cuối là bao nhiêu?

Hãy suy nghĩ từng bước trong thẻ <thinking>, sau đó đưa đáp án cuối trong thẻ <answer>.
""".strip()

print(chat(prompt))

In [ ]:
# Cách hiện đại: bật adaptive thinking (Claude tự suy nghĩ sâu, không cần thẻ)
response = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    thinking={"type": "adaptive"},
    messages=[{"role": "user", "content":
        "Một cửa hàng giảm 20%, rồi giảm thêm 10% trên giá đã giảm. "
        "Áo giá gốc 500.000đ thì giá cuối bao nhiêu?"}],
)

# Khi bật thinking, response.content có thể chứa block 'thinking' và 'text'
for block in response.content:
    if block.type == "thinking":
        print("[SUY NGHĨ]:", block.thinking[:200], "...\n")
    elif block.type == "text":
        print("[TRẢ LỜI]:", block.text)

## 6. Prefill — Mồi câu trả lời ⚠️

**Khóa cũ** dạy kỹ thuật *prefill*: thêm 1 message `assistant` để "mồi" đầu câu trả lời (ví dụ ép Claude bắt đầu bằng `{` để ra JSON, hoặc bỏ lời dẫn).

```python
# CÁCH CŨ (KHÔNG còn chạy trên Opus 4.8 — sẽ lỗi 400!)
messages=[
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "{"},   # mồi
]
```

➡️ **Trên model mới (Opus 4.6+, Sonnet 4.6), prefill ĐÃ BỊ BỎ.** Thay vào đó:
- Muốn **JSON có cấu trúc** → dùng `output_config` (xem ô dưới).
- Muốn **bỏ lời dẫn** ("Đây là...") → ra lệnh thẳng trong prompt: *"Trả lời trực tiếp, không mở đầu."*


In [ ]:
# Thay thế prefill bằng structured output để lấy JSON chuẩn
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content":
        "Trích xuất thông tin: Anh Nam, 28 tuổi, email nam@email.com, gói Premium."}],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "ten": {"type": "string"},
                    "tuoi": {"type": "integer"},
                    "email": {"type": "string"},
                    "goi": {"type": "string"},
                },
                "required": ["ten", "tuoi", "email", "goi"],
                "additionalProperties": False,
            },
        }
    },
)
print(response.content[0].text)

## 7. Chia nhỏ tác vụ (Prompt Chaining)

Với tác vụ lớn/phức tạp, **đừng nhồi tất cả vào 1 prompt**. Hãy chia thành chuỗi bước nhỏ, output của bước trước làm input cho bước sau → mỗi bước Claude tập trung làm tốt 1 việc.

Ví dụ: viết bài blog = (1) lập dàn ý → (2) viết bài theo dàn ý → (3) biên tập lại.


In [ ]:
# Bước 1: Lập dàn ý
dan_y = chat("Lập dàn ý ngắn (3 mục) cho bài blog về lợi ích của việc đọc sách.")
print("=== DÀN Ý ===")
print(dan_y)

# Bước 2: Viết bài dựa trên dàn ý (output bước 1 -> input bước 2)
bai_viet = chat(f"Dựa vào dàn ý sau, viết một đoạn blog ngắn (~120 từ):\n\n{dan_y}")
print("\n=== BÀI VIẾT ===")
print(bai_viet)

---
## ✅ Tóm tắt các kỹ thuật

| # | Kỹ thuật | Dùng khi |
|---|---|---|
| 1 | **Rõ ràng & trực tiếp** | Luôn luôn — nền tảng của mọi prompt |
| 2 | **Gán vai trò** (system) | Cần giọng văn/chuyên môn cụ thể |
| 3 | **Ví dụ (few-shot)** | Cần định dạng/phong cách ổn định |
| 4 | **Thẻ XML** | Prompt có nhiều phần dữ liệu |
| 5 | **Chain of thought** | Bài toán suy luận phức tạp |
| 6 | **Structured output** | Cần JSON (thay cho prefill cũ) |
| 7 | **Prompt chaining** | Tác vụ lớn, nhiều bước |

> 💡 Kết hợp các kỹ thuật này + **Prompt Evaluation** để đo lường và cải tiến liên tục.

**Khác khóa cũ:** Prefill (`assistant` mồi `{`) đã bị bỏ trên Opus 4.8 → dùng `output_config`. `temperature` cũng bị bỏ → dùng `output_config={"effort": ...}`.
